# ATC Multi-Agent GRPO Training

Trains **5 agents** (AMAN · DMAN · GENERATOR · SUPERVISOR · **ADAPT**) with GRPO on a live multi-agent air traffic control environment.

**Two training modes:**
- **LoRA** (default) — 4-bit QLoRA, runs on T4 · L4 · A100
- **Full-model** (`FULL_MODEL = True`) — whole-model fine-tuning with LLRD, needs A100/H100

**Long-horizon mode** (`LONG_HORIZON = True`) — multi-epoch planning episodes (Theme #2)

**Recommended runtime:** `Runtime → Change runtime type → GPU`  
T4 works for LoRA (50–100 episodes). A100 for full-model or longer runs.

## 1. Configuration

In [ ]:
from pathlib import Path
import os

# ── Repository ────────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/<your-user>/<your-repo>.git"   # ← set this
REPO_DIR  = Path("/content/ATC")

# ── Output ────────────────────────────────────────────────────────────────────
USE_DRIVE  = True
OUTPUT_DIR = (
    Path("/content/drive/MyDrive/atc-multiagent")
    if USE_DRIVE else
    Path("/content/atc-multiagent")
)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# ── Training mode ─────────────────────────────────────────────────────────────
FULL_MODEL   = False   # True → whole-model fine-tuning (needs A100/H100)
LONG_HORIZON = False   # True → multi-epoch long-horizon episodes (Theme #2)

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPISODES      = 50     # total training episodes (increase to 200+ for real training)
LORA_RANK     = 16     # LoRA rank (ignored when FULL_MODEL=True)
N_GENERATIONS = 4      # GRPO group size (min 4 for stable advantage variance)
SEED          = 42
EVAL_EPISODES = 5

# ── Environment ───────────────────────────────────────────────────────────────
os.environ["WANDB_MODE"]              = "disabled"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTHONUNBUFFERED"]        = "1"

print(f"REPO_URL      = {REPO_URL}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")
print(f"MODEL_NAME    = {MODEL_NAME}")
print(f"FULL_MODEL    = {FULL_MODEL}")
print(f"LONG_HORIZON  = {LONG_HORIZON}")
print(f"EPISODES      = {EPISODES}")
print(f"N_GENERATIONS = {N_GENERATIONS}")
print(f"LORA_RANK     = {LORA_RANK}")

## 2. Mount Drive and clone repo

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output ready: {OUTPUT_DIR}")

In [ ]:
import shutil, subprocess, sys

if "<your-user>" in REPO_URL or "<your-repo>" in REPO_URL:
    raise ValueError("Set REPO_URL in the config cell before cloning.")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

## 3. Install dependencies

In [ ]:
import subprocess, sys

def pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip("-U", "pip")
pip(
    "unsloth[colab-new]",
    "trl==0.9.6",
    "datasets>=2.20.0",
    "accelerate>=0.32.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.9.0",
    "numpy>=1.26.0",
    "openenv-core[core]>=0.2.3",
    "fastapi>=0.128.0",
    "openai>=2.30.0",
    "pydantic>=2.12.0",
    "uvicorn>=0.41.0",
)

print("Dependencies installed.")

## 4. Sanity check — GPU, dataset, all 5 agent roles

In [ ]:
import torch
from training.dataset import build_episode_dataset

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Build tiny sample to verify all 5 roles are present
sample = build_episode_dataset(
    n_episodes=4,
    seed=SEED,
    include_generator=True,
    include_supervisor=True,
    include_adapt=True,     # ADAPT agent included
    domain_episode_ratio=0.30,
)
roles_found = sorted({r["agent_role"] for r in sample})
print(f"\nDataset samples: {len(sample)}")
print(f"Roles found:     {roles_found}")
assert "ADAPT" in roles_found, "ADAPT role missing — check include_adapt=True"
print("All 5 agent roles verified: AMAN · DMAN · GENERATOR · SUPERVISOR · ADAPT")

## 5. Train — live reward tracking across all 5 roles

The training script emits `[ATC_REWARD step=N]` lines every 20 reward calls.  
This cell parses them in real time and displays an ASCII dashboard per role.

**Roles trained:**
- `AMAN` — Arrival Manager (temporal credit + hierarchical + recovery)
- `DMAN` — Departure Manager (ATFM compliance + same loss components)
- `GENERATOR` — Self-adapting curriculum (skill-gap-weighted regret)
- `SUPERVISOR` — Preference-following with calibration bonus
- `ADAPT` — Domain transfer meta-agent (downstream composite signal)

In [ ]:
import time, subprocess, sys, os, re
from collections import defaultdict

# ── Build training command ────────────────────────────────────────────────────
train_cmd = [
    sys.executable, "training/train_grpo.py",
    "--model",         MODEL_NAME,
    "--output_dir",    str(OUTPUT_DIR),
    "--episodes",      str(EPISODES),
    "--n_generations", str(N_GENERATIONS),
    "--lora_rank",     str(LORA_RANK),
    "--seed",          str(SEED),
]
if FULL_MODEL:
    train_cmd.append("--full_model")
if LONG_HORIZON:
    train_cmd.append("--long_horizon")

# ── Live reward tracking ──────────────────────────────────────────────────────
# Training script prints: [ATC_REWARD step=N] AMAN=0.xxxx DMAN=0.xxxx ... composite=0.xxxx
_ALL_ROLES    = ("AMAN", "DMAN", "GENERATOR", "SUPERVISOR", "ADAPT", "composite")
reward_history = defaultdict(list)
loss_history   = []
step_history   = []

_re_atc  = re.compile(r"\[ATC_REWARD step=(\d+)\] (.+)")
_re_kv   = re.compile(r"(\w+)=([\-\d.eE+]+)")
_re_loss = re.compile(r"'loss':\s*([\-\d.eE+]+)")
_re_step = re.compile(r"'step':\s*(\d+)")

def _bar(val, lo=-1.0, hi=1.0, width=24):
    frac   = max(0.0, min(1.0, (val - lo) / max(1e-9, hi - lo)))
    filled = int(frac * width)
    colour = "▓" if val >= 0 else "░"
    return f"[{colour * filled}{'-' * (width - filled)}] {val:+.3f}"

def _trend(vals):
    if len(vals) < 40:
        return ""
    half  = len(vals) // 2
    early = sum(vals[:half]) / half
    late  = sum(vals[half:]) / max(1, len(vals) - half)
    return "  ↑" if late > early + 0.03 else ("  ↓" if late < early - 0.03 else "  →")

def _live_dashboard(step, elapsed_s):
    print(f"\n{'─'*56}")
    print(f"  step {step:5d}  │  elapsed {elapsed_s/60:.1f} min")
    print(f"{'─'*56}")
    if loss_history:
        print(f"  {'loss':12s}: {loss_history[-1]:.5f}")
    for role in _ALL_ROLES:
        vals = reward_history.get(role, [])
        if not vals:
            continue
        recent  = vals[-min(20, len(vals)):]
        mean_r  = sum(recent) / len(recent)
        print(f"  {role:12s}: {_bar(mean_r)}{_trend(vals)}")
    print()

# ── Launch ────────────────────────────────────────────────────────────────────
env_copy = os.environ.copy()
env_copy["PYTHONUNBUFFERED"] = "1"

print("Command:", " ".join(train_cmd))
print("=" * 60)
t0           = time.time()
last_dash    = -1

proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env_copy,
)

for raw_line in proc.stdout:
    line = raw_line.rstrip("\n")
    print(line, flush=True)

    # Parse [ATC_REWARD step=N] lines emitted by our RewardLogger
    m_atc = _re_atc.match(line)
    if m_atc:
        step = int(m_atc.group(1))
        for kv in _re_kv.finditer(m_atc.group(2)):
            reward_history[kv.group(1)].append(float(kv.group(2)))
        if step != last_dash:
            last_dash = step
            step_history.append(step)
            _live_dashboard(step, time.time() - t0)
        continue

    # Also parse TRL dict-format lines for loss
    if "'loss'" in line:
        m_loss = _re_loss.search(line)
        if m_loss:
            loss_history.append(float(m_loss.group(1)))

proc.wait()
elapsed = time.time() - t0
print("=" * 60)

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n=== FINAL REWARD SUMMARY ===")
for role in _ALL_ROLES:
    vals = reward_history.get(role, [])
    if not vals:
        continue
    n    = len(vals)
    avg  = sum(vals) / n
    fq   = sum(vals[:max(1, n // 4)]) / max(1, n // 4)
    lq   = sum(vals[max(0, 3 * n // 4):]) / max(1, n - 3 * n // 4)
    trend = "↑" if lq > fq + 0.03 else ("↓" if lq < fq - 0.03 else "→")
    print(f"  {role:12s}: mean={avg:+.3f}  first_q={fq:+.3f}  last_q={lq:+.3f}  {trend}")

status = "complete" if proc.returncode == 0 else f"FAILED (exit code {proc.returncode})"
print(f"\nTraining {status} in {elapsed / 60:.1f} min")

## 6. Evaluate trained model vs heuristic baseline

In [ ]:
import subprocess, sys

eval_json = OUTPUT_DIR / "eval_results.json"
eval_cmd  = [
    sys.executable, "training/eval.py",
    "--base",     "heuristic-baseline",
    "--trained",  str(OUTPUT_DIR),
    "--episodes", str(EVAL_EPISODES),
    "--output",   str(eval_json),
]

print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)
print(f"Saved eval results to {eval_json}")

## 7. Plot reward curves — all 5 roles

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

curves_path = OUTPUT_DIR / "reward_curves.json"
plots_dir   = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

with open(curves_path) as f:
    curves = json.load(f)

ROLE_COLOURS = {
    "AMAN":       "#2196F3",   # blue
    "DMAN":       "#4CAF50",   # green
    "GENERATOR":  "#F44336",   # red
    "SUPERVISOR": "#FF9800",   # orange
    "ADAPT":      "#9C27B0",   # purple
    "composite":  "#212121",   # dark
}

def _smooth(vals, w=20):
    if len(vals) < w:
        return vals
    return [sum(vals[max(0,i-w):i+1]) / min(w, i+1) for i in range(len(vals))]

# ── Plot 1: per-role reward curves ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("ATC Multi-Agent GRPO — Per-Role Reward Curves", fontsize=14, fontweight="bold")

roles_plot = ["AMAN", "DMAN", "GENERATOR", "SUPERVISOR", "ADAPT", "composite"]
for ax, role in zip(axes.flat, roles_plot):
    vals = curves.get(role, [])
    if not vals:
        ax.set_title(f"{role} (no data)")
        continue
    x    = list(range(len(vals)))
    col  = ROLE_COLOURS.get(role, "#607D8B")
    ax.plot(x, vals, alpha=0.25, color=col, linewidth=0.8, label="raw")
    ax.plot(x, _smooth(vals), color=col, linewidth=2.0, label="smoothed (w=20)")
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(f"{role}", fontweight="bold", color=col)
    ax.set_xlabel("Training call")
    ax.set_ylabel("Reward")
    ax.set_ylim(-1.05, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    # Annotate final mean
    tail = vals[-max(1, len(vals)//4):]
    ax.axhline(sum(tail)/len(tail), color=col, linewidth=1.0, linestyle=":")
    ax.text(0.98, sum(tail)/len(tail) + 0.05, f"final mean={sum(tail)/len(tail):+.3f}",
            ha="right", va="bottom", fontsize=8, transform=ax.get_yaxis_transform(), color=col)

plt.tight_layout()
out_path = plots_dir / "training_curves_all_roles.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

# ── Plot 2: composite reward with first/last quartile comparison ──────────────
comp = curves.get("composite", [])
if comp:
    fig2, ax2 = plt.subplots(figsize=(12, 4))
    x = list(range(len(comp)))
    ax2.plot(x, comp, alpha=0.2, color="#212121", linewidth=0.7)
    ax2.plot(x, _smooth(comp, 30), color="#212121", linewidth=2.2, label="smoothed")
    # Quartile bands
    q1_end = max(1, len(comp) // 4)
    q4_start = max(0, 3 * len(comp) // 4)
    q1_mean = sum(comp[:q1_end]) / q1_end
    q4_mean = sum(comp[q4_start:]) / max(1, len(comp) - q4_start)
    ax2.axhspan(-1, q1_mean, xmin=0, xmax=0.25, alpha=0.08, color="red", label=f"Q1 mean={q1_mean:+.3f}")
    ax2.axhspan(-1, q4_mean, xmin=0.75, xmax=1.0, alpha=0.08, color="green", label=f"Q4 mean={q4_mean:+.3f}")
    ax2.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax2.set_title("Composite Reward — Training Progress", fontweight="bold")
    ax2.set_xlabel("Training call")
    ax2.set_ylabel("Composite reward")
    ax2.set_ylim(-1.05, 1.05)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    out2 = plots_dir / "composite_reward_progress.png"
    plt.tight_layout()
    plt.savefig(out2, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out2}")
    delta = q4_mean - q1_mean
    print(f"\nImprovement Q1→Q4: {delta:+.3f} ({'↑ learning' if delta > 0.03 else '→ stable' if delta > -0.03 else '↓ degraded'})")

## 8. Before / After comparison

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

baseline_path = OUTPUT_DIR / "baseline_metrics.json"
trained_path  = OUTPUT_DIR / "trained_model_metrics.json"

if not baseline_path.exists() or not trained_path.exists():
    print("Eval files not found — run the evaluation cell first.")
else:
    with open(baseline_path) as f: baseline = json.load(f)
    with open(trained_path)  as f: trained  = json.load(f)

    metrics = [
        ("mean_composite",   "Composite score"),
        ("mean_aman_reward", "AMAN reward"),
        ("mean_dman_reward", "DMAN reward"),
        ("mean_conflicts",   "Avg conflicts (lower=better)"),
        ("mean_emg_handled", "Emergencies handled"),
    ]

    labels   = [m[1] for m in metrics]
    before   = [baseline.get(m[0], 0) for m in metrics]
    after    = [trained.get(m[0], 0)  for m in metrics]

    x      = np.arange(len(labels))
    width  = 0.35
    fig, ax = plt.subplots(figsize=(12, 5))
    bars_b  = ax.bar(x - width/2, before, width, label=baseline.get("tag", "Baseline"), color="#90A4AE")
    bars_a  = ax.bar(x + width/2, after,  width, label=trained.get("tag",  "Trained"),  color="#2196F3")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Before vs. After GRPO Training — All 5 Agent Roles", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    # Delta annotations
    for b, a, xi in zip(before, after, x):
        d = a - b
        ax.text(xi + width/2, max(a, b) + 0.02, f"{d:+.3f}",
                ha="center", va="bottom", fontsize=9,
                color="green" if d > 0 else "red")
    plt.tight_layout()
    out3 = plots_dir / "before_after_comparison.png"
    plt.savefig(out3, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out3}")

    print("\n=== BEFORE vs AFTER ===")
    for key, label in metrics:
        b = baseline.get(key, 0)
        a = trained.get(key, 0)
        d = a - b
        arrow = "↑" if d > 0.005 else ("↓" if d < -0.005 else "→")
        print(f"  {label:32s}: {b:6.3f}  →  {a:6.3f}  ({d:+.3f} {arrow})")

## Troubleshooting

| Problem | Fix |
|---|---|
| OOM on T4 | Reduce `EPISODES`, or set `LORA_RANK=8` |
| `GRPOConfig` keyword error | Re-run the install cell — needs `trl==0.9.6` |
| ADAPT role missing from dashboard | Re-run sanity check cell to verify `include_adapt=True` |
| `--full_model` on T4 | Needs A100/H100 — 7B full precision won't fit in 16 GB VRAM |
| Colab disconnects | Keep `OUTPUT_DIR` on Drive so checkpoints survive |
| All rewards stuck at −1.0 | Check that `REPO_URL` points to the correct branch with reward functions |